In [5]:
!pip install -q "transformers==4.46.3" "datasets>=2.20" "peft>=0.13" "accelerate>=1.0" "seqeval" "evaluate" "scikit-learn"
!pip install -U torchao
!pip install boto3

In [6]:
import json
import os
import shutil
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import boto3
import kagglehub
import numpy as np
import torch
from google.colab import userdata
from datasets import Dataset, DatasetDict
from peft import LoraConfig, TaskType, get_peft_model
from seqeval.metrics import classification_report as seq_report
from seqeval.metrics import f1_score as seq_f1
from seqeval.metrics import precision_score as seq_p
from seqeval.metrics import recall_score as seq_r
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Load Secrets from Google Colab
AWS_ACCESS_KEY = userdata.get('AWS_ACCESS_KEY')
AWS_SECRET_KEY = userdata.get('AWS_SECRET_KEY')
BUCKET_NAME = userdata.get('S3_BUCKET_NAME')

# Path to the dataset file downloaded from Kaggle.
# The Kaggle dataset ships as a single JSON list of {text, annotations}.
DATA_PATH = kagglehub.dataset_download("yashpwrr/resume-ner-training-dataset") + "/train.json"

# DeBERTa-v3-base gives the best F1 for NER at this size.
# If you hit tokenizer / sentencepiece issues on Colab, fall back to
# "bert-base-cased" — code below works with either.
MODEL_NAME = "bert-base-cased"

OUTPUT_DIR = "resume-ner-lora"
MAX_LEN = 384  # resumes are long; 384 covers most without truncating too much
BATCH_SIZE = 8
EPOCHS = 6
LR = 2e-4  # higher LR is fine because LoRA only trains adapters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Labels we actually want. Everything else gets dropped.
# These are the ones that are (a) useful for the resume schema and
# (b) reliably annotated in the dataset.
KEEP_LABELS = [
    "PERSON",
    "EMAIL",
    "DESIGNATION",
    "EDUCATION",
    "LOCATION"
]

100%|██████████| 14.2M/14.2M [00:00<00:00, 97.9MB/s]

Extracting files...


In [7]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Loaded {len(raw)} samples")
print("Sample keys:", list(raw[0].keys()))
print("Example annotations:", raw[0]["annotations"][:3])

Loaded 5960 samples
Sample keys: ['text', 'annotations']
Example annotations: [[1296, 1622, 'SKILL'], [993, 1154, 'SKILL'], [939, 957, 'PERSON']]


In [8]:
label_counts = Counter()
for ex in raw:
    for ann in ex["annotations"]:
        # Each annotation is [start, end, label]
        if len(ann) >= 3:
            label_counts[ann[2]] += 1

print("Label distribution in raw data:")
for lbl, cnt in label_counts.most_common():
    keep = "KEEP" if lbl in KEEP_LABELS else "drop"
    print(f"  {lbl:15s} {cnt:>8d}  [{keep}]")

Label distribution in raw data:
  SKILL             549465  [drop]
  OTHER              10267  [drop]
  DESIGNATION         4301  [KEEP]
  LOCATION            4073  [KEEP]
  EXPERIENCE          3544  [drop]
  PERSON              3122  [KEEP]
  EDUCATION           2124  [KEEP]
  EXPERTISE           1045  [drop]
  EMAIL                815  [KEEP]
  COMPANY              218  [drop]
  COLLABORATION        187  [drop]
  LANGUAGE             159  [drop]
  ACTION               133  [drop]
  CERTIFICATION        122  [drop]


In [9]:
def clean_annotations(text, anns):
    """
    1. Drop labels not in KEEP_LABELS.
    2. Drop spans whose offsets are clearly broken (out of range, end<=start).
    3. Trim leading/trailing whitespace and adjust offsets.
    4. Resolve overlapping spans by keeping the LONGEST one
       (longest = most specific in this dataset's noise pattern).
    """
    cleaned = []
    for ann in anns:
        if len(ann) < 3:
            continue
        start, end, label = ann[0], ann[1], ann[2]
        if label not in KEEP_LABELS:
            continue
        if not (0 <= start < end <= len(text)):
            continue
        span = text[start:end]
        # Trim whitespace
        lead = len(span) - len(span.lstrip())
        trail = len(span) - len(span.rstrip())
        new_start = start + lead
        new_end = end - trail
        if new_end <= new_start:
            continue
        cleaned.append((new_start, new_end, label))

    # Resolve overlaps: sort by length desc, accept span if it does not overlap
    # any already-accepted span.
    cleaned.sort(key=lambda x: (x[1] - x[0]), reverse=True)
    accepted = []
    for s, e, l in cleaned:
        overlap = any(not (e <= a_s or s >= a_e) for a_s, a_e, _ in accepted)
        if not overlap:
            accepted.append((s, e, l))
    accepted.sort(key=lambda x: x[0])
    return accepted

cleaned_data = []
for ex in raw:
    text = ex["text"]
    anns = clean_annotations(text, ex["annotations"])
    if not anns:
        # skip docs with no usable labels — they hurt training
        continue
    cleaned_data.append({"text": text, "spans": anns})

print(f"Kept {len(cleaned_data)} / {len(raw)} documents after cleaning")

Kept 988 / 5960 documents after cleaning


In [10]:
# Sanity-check class balance after filtering. Run this once and look at
# the printed counts -- if any label has < 30 examples, drop it too.
from collections import Counter  # noqa: E402

label_counts = Counter()
for ex in cleaned_data:                     # `cleaned` is the filtered dataset from CELL 4
    for _, _, lbl in ex["spans"]:
        label_counts[lbl] += 1

print("Entity counts after filtering:")
for lbl, n in label_counts.most_common():
    print(f"  {lbl:<15} {n}")

Entity counts after filtering:
  DESIGNATION     4293
  LOCATION        4069
  PERSON          3118
  EDUCATION       2119
  EMAIL           815


In [11]:
LABEL_LIST = ["O"]
for lbl in sorted(KEEP_LABELS):
    LABEL_LIST.append(f"B-{lbl}")
    LABEL_LIST.append(f"I-{lbl}")
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
print(f"{len(LABEL_LIST)} BIO labels")

11 BIO labels


In [12]:
train_data, test_data = train_test_split(
    cleaned_data, test_size=0.10, random_state=SEED
)
train_data, val_data = train_test_split(
    train_data, test_size=0.10, random_state=SEED
)
print(f"Train: {len(train_data)}  Val: {len(val_data)}  Test: {len(test_data)}")

Train: 800  Val: 89  Test: 99


In [13]:
from transformers import AutoTokenizer  # noqa: E402

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def encode_example(example):
    text = example["text"]

    text = text.encode("utf-8", errors="ignore").decode("utf-8")

    spans = example["spans"]

    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LEN,
        return_offsets_mapping=True,
        return_special_tokens_mask=True,
    )
    offsets = enc.pop("offset_mapping")
    special = enc.pop("special_tokens_mask")

    labels = []
    span_idx = 0
    inside_span = None
    for (tok_start, tok_end), is_special in zip(offsets, special):
        if is_special or tok_start == tok_end:
            labels.append(-100)
            continue

        while span_idx < len(spans) and spans[span_idx][1] <= tok_start:
            span_idx += 1

        assigned = False
        if span_idx < len(spans):
            s, e, lbl = spans[span_idx]
            # token strictly inside or substantially overlapping a span
            if tok_end > s and tok_start < e:
                if inside_span != (s, e, lbl):
                    labels.append(LABEL2ID[f"B-{lbl}"])
                    inside_span = (s, e, lbl)
                else:
                    labels.append(LABEL2ID[f"I-{lbl}"])
                assigned = True
        if not assigned:
            labels.append(LABEL2ID["O"])
            inside_span = None

    enc["labels"] = labels
    return enc


def to_hf_dataset(rows):
    return Dataset.from_list([encode_example(r) for r in rows])


train_ds = to_hf_dataset(train_data)
val_ds = to_hf_dataset(val_data)
test_ds = to_hf_dataset(test_data)

dsd = DatasetDict(train=train_ds, validation=val_ds, test=test_ds)
print(dsd)


# Run this to *visually verify* the BIO tags line up with the text.
# If you see entities mis-aligned here, ALL training will fail. This
# is the single most important debugging step for token-classification.
from transformers import AutoTokenizer as _AT  # noqa: E402

_sample = train_ds[0]
_tokens = tokenizer.convert_ids_to_tokens(_sample["input_ids"])
print(f"{'TOKEN':<20s} {'LABEL'}")
print("-" * 40)
for tok, lid in zip(_tokens[:80], _sample["labels"][:80]):
    lbl = ID2LABEL[lid] if lid != -100 else "IGN"
    print(f"{tok:<20s} {lbl}")
# You should see, e.g., "John" -> B-PERSON, "Smith" -> I-PERSON, etc.
# If everything is "O", your spans and tokenization don't agree.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 89
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 99
    })
})
TOKEN                LABEL
----------------------------------------
[CLS]                IGN
"                    O
j                    O
##H                  O
Joseph               B-PERSON
C                    I-PERSON
.                    I-PERSON
Herr                 I-PERSON
##on                 I-PERSON
Senior               B-DESIGNATION
U                    I-DESIGNATION
##X                  I-DESIGNATION
,                    I-DESIGNATION
U                    I-DESIGNATION
##I                  I-DESIGNATION
and                  I-DESIGNATION
Visual               I-DESIGNATION
De

In [14]:
# THE critical fix: `modules_to_save=["classifier"]` so the classification
# head is actually trained. Without this, LoRA adapters feed into a frozen
# RANDOM head and loss diverges.
import torch  # noqa: E402
from peft import LoraConfig, TaskType, get_peft_model  # noqa: E402
from transformers import AutoModelForTokenClassification  # noqa: E402

base_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

lora_cfg = LoraConfig(
    task_type=TaskType.TOKEN_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    # BERT attention modules. PEFT silently ignores names that don't exist,
    # so this list works for both BERT and DeBERTa-v3 if you swap MODEL_NAME.
    target_modules=["query", "key", "value",
                    "query_proj", "key_proj", "value_proj"],
    # *** THIS IS THE FIX *** — train the classifier head fully.
    modules_to_save=["classifier"],
)

model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()
# Look for "trainable params: ~1.5M" or similar. If you see <100k trainable,
# modules_to_save didn't kick in and you'll get the same broken result.

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 893,195 || all params: 108,621,334 || trainable%: 0.8223


In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_seq, pred_seq = [], []
    for p_row, l_row in zip(preds, labels):
        t, pr = [], []
        for p_id, l_id in zip(p_row, l_row):
            if l_id == -100:
                continue
            t.append(ID2LABEL[int(l_id)])
            pr.append(ID2LABEL[int(p_id)])
        true_seq.append(t)
        pred_seq.append(pr)

    return {
        "precision": seq_p(true_seq, pred_seq),
        "recall": seq_r(true_seq, pred_seq),
        "f1": seq_f1(true_seq, pred_seq),
    }

In [16]:
# Changes:
#  - fp16 / bf16 BOTH OFF. T4 doesn't support bf16 natively, and fp16 + LoRA
#    + token classification is the combo that gave you the unscale error.
#    fp32 is ~2x slower but rock solid. With 5K samples it's fine.
#  - `processing_class=tokenizer` is the new API (was `tokenizer=tokenizer`)
#  - `gradient_checkpointing=True` to fit in T4 memory at fp32
#  - lower LR (2e-4) — set in CELL 2 above
#  - max_grad_norm=1.0 to prevent the gradient explosion you saw
from transformers import (  # noqa: E402
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=False,
    bf16=False,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    seed=SEED,
)

# gradient checkpointing requires this for PEFT models
model.enable_input_require_grads()

collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dsd["train"],
    eval_dataset=dsd["validation"],
    processing_class=tokenizer,   # new API; was tokenizer=tokenizer
    data_collator=collator,
    compute_metrics=compute_metrics,
)


In [17]:
baseline_metrics = trainer.evaluate(dsd["test"])
print("BASELINE (pre-fine-tune) test metrics:", baseline_metrics)

BASELINE (pre-fine-tune) test metrics: {'eval_loss': 2.4424216747283936, 'eval_model_preparation_time': 0.0296, 'eval_precision': 0.0006030243992949253, 'eval_recall': 0.016352201257861635, 'eval_f1': 0.0011631548338030691, 'eval_runtime': 3.992, 'eval_samples_per_second': 24.8, 'eval_steps_per_second': 1.754}


In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Precision,Recall,F1
1,0.470000,0.378971,0.029600,0.068921,0.068653,0.068787
2,0.244100,0.231284,0.029600,0.263626,0.306995,0.283662
3,0.220300,0.195738,0.029600,0.355088,0.415803,0.383055
4,0.196200,0.184026,0.029600,0.380000,0.492228,0.428894
5,0.177600,0.179039,0.029600,0.388060,0.505181,0.438942
6,0.167900,0.178549,0.029600,0.392315,0.502591,0.440659


TrainOutput(global_step=600, training_loss=0.3628593810399373, metrics={'train_runtime': 457.3009, 'train_samples_per_second': 10.496, 'train_steps_per_second': 1.312, 'total_flos': 950622887116800.0, 'train_loss': 0.3628593810399373, 'epoch': 6.0})

In [19]:
finetuned_metrics = trainer.evaluate(dsd["test"])
print("FINE-TUNED test metrics:", finetuned_metrics)

# Per-entity classification report
preds_out = trainer.predict(dsd["test"])
preds = np.argmax(preds_out.predictions, axis=-1)
labels = preds_out.label_ids

true_seq, pred_seq = [], []
for p_row, l_row in zip(preds, labels):
    t, pr = [], []
    for p_id, l_id in zip(p_row, l_row):
        if l_id == -100:
            continue
        t.append(ID2LABEL[int(l_id)])
        pr.append(ID2LABEL[int(p_id)])
    true_seq.append(t)
    pred_seq.append(pr)

print(seq_report(true_seq, pred_seq, digits=4))

FINE-TUNED test metrics: {'eval_loss': 0.18818919360637665, 'eval_model_preparation_time': 0.0296, 'eval_precision': 0.3829174664107486, 'eval_recall': 0.5018867924528302, 'eval_f1': 0.4344039194338596, 'eval_runtime': 2.9822, 'eval_samples_per_second': 33.197, 'eval_steps_per_second': 2.347, 'epoch': 6.0}
              precision    recall  f1-score   support

 DESIGNATION     0.3615    0.5144    0.4246       208
   EDUCATION     0.3830    0.4655    0.4202       116
       EMAIL     0.7711    0.9552    0.8533        67
    LOCATION     0.3147    0.3575    0.3347       221
      PERSON     0.3506    0.5191    0.4185       183

   micro avg     0.3829    0.5019    0.4344       795
   macro avg     0.4362    0.5624    0.4903       795
weighted avg     0.3836    0.5019    0.4337       795



In [20]:
# # Option A: keep LoRA adapters separate (smaller, recommended)
model.save_pretrained(f"{OUTPUT_DIR}/adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")

shutil.make_archive("/content/resume_ner_adapter", "zip", f"{OUTPUT_DIR}/adapter")
print(os.path.getsize("/content/resume_ner_adapter.zip") / 1e6, "MB")

3.631432 MB


In [21]:
merged = model.merge_and_unload()
merged.save_pretrained(f"{OUTPUT_DIR}/merged")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged")

shutil.make_archive("/content/resume_ner_merged", "zip", f"{OUTPUT_DIR}/merged")
print(os.path.getsize("/content/resume_ner_merged.zip") / 1e6, "MB")

print("Uploading to S3...")
s3 = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)
s3.upload_file("/content/resume_ner_merged.zip", BUCKET_NAME, "models/resume_ner_merged.zip")

print("✅ Model successfully uploaded to S3!")

400.266194 MB
Uploading to S3...
✅ Model successfully uploaded to S3!


In [22]:
import os
adapter_size = sum(os.path.getsize(os.path.join(f"{OUTPUT_DIR}/adapter", f))
                   for f in os.listdir(f"{OUTPUT_DIR}/adapter")) / 1e6
merged_size  = sum(os.path.getsize(os.path.join(f"{OUTPUT_DIR}/merged", f))
                   for f in os.listdir(f"{OUTPUT_DIR}/merged")) / 1e6
print(f"Adapter: {adapter_size:.1f} MB | Merged: {merged_size:.1f} MB | "
      f"LoRA stores {adapter_size/merged_size*100:.1f}% of the model size")

Adapter: 4.5 MB | Merged: 431.8 MB | LoRA stores 1.0% of the model size


In [23]:
import re
from transformers import pipeline

# Reload the merged fine-tuned model for inference
ner = pipeline(
    "token-classification",
    model=f"{OUTPUT_DIR}/merged",
    tokenizer=f"{OUTPUT_DIR}/merged",
    aggregation_strategy="simple",   # merges B-/I- subwords for us
    device=0 if torch.cuda.is_available() else -1,
)

# --- Regex layer (high-precision contact info) ---
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
# Phone: requires at least 7 digits, allows +, spaces, dashes, parens
PHONE_RE = re.compile(
    r"(?:\+?\d{1,3}[\s\-.])?(?:\(?\d{2,4}\)?[\s\-.]?)\d{3,4}[\s\-.]?\d{3,4}"
)
# Reject phone candidates that look like year ranges or pure dates
DATE_LIKE_RE = re.compile(r"^\s*(19|20)\d{2}\s*[-/]\s*(19|20)?\d{2}\s*$")

EDU_KEYWORDS = (
    "university", "college", "institute", "school", "academy",
    "polytechnic", "iit", "iim", "nit", "high school"
)

# Tiny built-in skill taxonomy. Replace with ESCO/O*NET subset for the real demo.
SKILL_DICT = {
    "python","java","javascript","typescript","c++","c#","go","rust","ruby","php","scala","kotlin","swift",
    "react","angular","vue","next.js","node.js","express","django","flask","fastapi","spring","rails",
    "tensorflow","pytorch","keras","scikit-learn","pandas","numpy","matplotlib",
    "aws","gcp","azure","docker","kubernetes","terraform","ansible","jenkins","git","github","gitlab",
    "postgresql","mysql","mongodb","redis","elasticsearch","cassandra","sqlite","oracle",
    "linux","bash","sql","nosql","graphql","rest","grpc","kafka","rabbitmq","spark","hadoop","airflow",
    "html","css","sass","tailwind","figma","photoshop",
    "machine learning","deep learning","nlp","computer vision","data analysis","statistics",
}

def extract_with_regex(text):
    """Pull EMAIL and PHONE with regex - 100% reliable, never trust the model here."""
    out = []
    for m in EMAIL_RE.finditer(text):
        out.append({"label": "EMAIL", "text": m.group(0).strip(),
                    "start": m.start(), "end": m.end(), "score": 1.0})
    for m in PHONE_RE.finditer(text):
        cand = m.group(0).strip()
        digits = re.sub(r"\D", "", cand)
        if len(digits) < 7 or len(digits) > 15:
            continue
        if DATE_LIKE_RE.match(cand):
            continue
        out.append({"label": "PHONE", "text": cand,
                    "start": m.start(), "end": m.end(), "score": 1.0})
    return out

def extract_skills_dict(text):
    """Dictionary lookup against a known skill taxonomy."""
    out = []
    low = text.lower()
    for skill in SKILL_DICT:
        for m in re.finditer(rf"(?<![a-z0-9+#.]){re.escape(skill)}(?![a-z0-9+#])", low):
            out.append({"label": "SKILL", "text": text[m.start():m.end()],
                        "start": m.start(), "end": m.end(), "score": 1.0})
    return out

def rescue_education(spans):
    """If a PERSON span contains a school keyword, reclassify as EDUCATION."""
    fixed = []
    for s in spans:
        low = s["text"].lower()
        if s["label"] == "PERSON" and any(k in low for k in EDU_KEYWORDS):
            s = {**s, "label": "EDUCATION"}
        fixed.append(s)
    return fixed

def merge_adjacent(spans, max_gap=3):
    """Merge same-label spans separated by <= max_gap chars (whitespace/punct)."""
    if not spans:
        return spans
    spans = sorted(spans, key=lambda x: x["start"])
    merged = [spans[0]]
    for s in spans[1:]:
        prev = merged[-1]
        if s["label"] == prev["label"] and 0 <= s["start"] - prev["end"] <= max_gap:
            merged[-1] = {**prev, "end": s["end"],
                          "text": prev["text"] + " " + s["text"]}
        else:
            merged.append(s)
    return merged

def dedupe(spans):
    """Drop exact (label, normalized text) duplicates, keep first."""
    seen, out = set(), []
    for s in spans:
        key = (s["label"], re.sub(r"\s+", " ", s["text"].strip().lower()))
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
    return out

def resolve_overlaps(spans):
    """Drop shorter spans when two overlap with different labels."""
    spans = sorted(spans, key=lambda s: (s["start"], -(s["end"] - s["start"])))
    keep = []
    for s in spans:
        if any(not (s["end"] <= k["start"] or s["start"] >= k["end"]) for k in keep):
            continue
        keep.append(s)
    return sorted(keep, key=lambda s: s["start"])

def predict_entities(text):
    text = text.encode("utf-8", errors="ignore").decode("utf-8")

    # 1. Model handles structural fields only
    raw = ner(text)
    model_spans = [
        {"label": e["entity_group"], "text": text[e["start"]:e["end"]],
         "start": int(e["start"]), "end": int(e["end"]), "score": float(e["score"])}
        for e in raw
        if e["entity_group"] in {"PERSON", "DESIGNATION", "EDUCATION", "LOCATION"}
        and e["score"] >= 0.55
    ]

    # 2. Apply post-processing chain
    model_spans = rescue_education(model_spans)
    model_spans = merge_adjacent(model_spans, max_gap=3)

    # 3. Add regex layer (email, phone) and dictionary layer (skills)
    regex_spans = extract_with_regex(text)
    skill_spans = extract_skills_dict(text)

    all_spans = model_spans + regex_spans + skill_spans
    all_spans = resolve_overlaps(all_spans)
    all_spans = dedupe(all_spans)
    return all_spans


In [24]:
DEMO = """John Smith
Email: john.smith@gmail.com  |  Phone: +1 415-555-0142
San Francisco, CA

EXPERIENCE
Senior Software Engineer, Acme Corp (2018-Present)
- Built distributed services in Python and Go on GCP and Kubernetes.

Software Engineer, Initech (2017-2018)
- Backend in Java with PostgreSQL.

EDUCATION
B.S. Computer Science, Stanford University, 2013-2017

SKILLS
Python, Go, Java, Kubernetes, Docker, AWS, GCP, PostgreSQL, React, TypeScript
"""

results = predict_entities(DEMO)

# Group by label for readable output
grouped = defaultdict(list)
for r in results:
    grouped[r["label"]].append(r["text"])

print("=" * 60)
print("STRUCTURED RESUME EXTRACTION")
print("=" * 60)
for label in ["PERSON", "EMAIL", "PHONE", "LOCATION",
              "DESIGNATION", "EDUCATION", "SKILL"]:
    vals = grouped.get(label, [])
    print(f"{label:13}: {vals if vals else '-'}")

STRUCTURED RESUME EXTRACTION
PERSON       : ['John Smith']
EMAIL        : ['john.smith@gmail.com']
PHONE        : ['+1 415-555-0142']
LOCATION     : ['San Francisco, CA']
DESIGNATION  : ['Senior Software Engineer', 'Software Engineer']
EDUCATION    : ['B.S. Computer Science']
SKILL        : ['Python', 'Go', 'GCP', 'Kubernetes', 'Java', 'PostgreSQL', 'Docker', 'AWS', 'React', 'TypeScript']
